In [1]:
!pip install spacy
!pip install scikit-learn
!pip install cltk
!pip install --upgrade numpy spacy cltk


  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)


In [2]:
import os
import numpy as np
from cltk import NLP
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.environ['CLTK_DATA'] = os.path.expanduser('~/cltk_data')
cltk_nlp = NLP(language="grc", suppress_banner=True)

#стоп-слова
ancient_greek_stopwords = []
if os.path.exists("stopwords.txt"):
    with open("stopwords.txt", "r", encoding="utf-8") as f:
        for line in f:
            word = line.strip()
            if word:
                ancient_greek_stopwords.append(word)

#словарь ручных правок
manual_lemmas = {}
if os.path.exists("lemmas.txt"):
    with open("lemmas.txt", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() and ":" in line:
                key, value = line.split(":", 1)
                manual_lemmas[key.strip().lower()] = value.strip().lower()

# лемматизация
def get_lemmas(text):
    if not text.strip():
        return ""
    doc = cltk_nlp.analyze(text=text)
    lemmas = []
    for word in doc.words:
        if not word.string:
            continue
        word_lower = word.string.lower()
        lemma = manual_lemmas.get(word_lower, word.lemma)
        if lemma:
            lemmas.append(lemma.lower())
    return " ".join(lemmas)

# корпус
file_names = {
    "josas1": "josas1.txt", "josas2": "josas2.txt",
    "jdt1": "jdt1.txt", "jdt2": "jdt2.txt",
    "3makk1": "3makk1.txt", "3makk2": "3makk2.txt",
    "wisd1": "wisd1.txt", "wisd2": "wisd2.txt",
    "testjob1": "testjob1.txt", "testjob2": "testjob2.txt"
}

cleaned_corpus = {}
for label, file_path in file_names.items():
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            cleaned_corpus[label] = get_lemmas(f.read())

# векторизация. расчет косинусного сходства
labels = list(cleaned_corpus.keys())
corpus = list(cleaned_corpus.values())

MFW_COUNT = 100

vectorizer = CountVectorizer(
    stop_words=ancient_greek_stopwords,
    token_pattern=r"\S+",
    max_features=MFW_COUNT
)
X_absolute = vectorizer.fit_transform(corpus).toarray()
similarity_matrix = cosine_similarity(X_absolute)

#матрица
print(f"\nМатрица косинусного сходства (MFW = {MFW_COUNT})")
header = "             " + " ".join([f"{l:<10}" for l in labels])
print(header)
for i, row in enumerate(similarity_matrix):
    row_str = " ".join([f"{val:<10.4f}" for val in row])
    print(f"{labels[i]:<12} {row_str}")

# интерпретация
print("\nРезультаты сравнения внутри памятников")
print(f"Иосиф и Асенет (1 vs 2):          {similarity_matrix[labels.index('josas1')][labels.index('josas2')]:.4f}")
print(f"Иудифь (1 vs 2):                  {similarity_matrix[labels.index('jdt1')][labels.index('jdt2')]:.4f}")
print(f"3 Маккавейская (1 vs 2):          {similarity_matrix[labels.index('3makk1')][labels.index('3makk2')]:.4f}")
print(f"Премудрость Соломона (1 vs 2):    {similarity_matrix[labels.index('wisd1')][labels.index('wisd2')]:.4f}")
print(f"Завещание Иова (1 vs 2):          {similarity_matrix[labels.index('testjob1')][labels.index('testjob2')]:.4f}")
print("\nСравнение текстов разных авторов")
print(f"Сравнение ИиА1 c Иудифь2:         {similarity_matrix[labels.index('josas1')][labels.index('jdt2')]:.4f}")

diff_pairs = [
    similarity_matrix[labels.index('3makk1')][labels.index('wisd1')], similarity_matrix[labels.index('3makk1')][labels.index('wisd2')],
    similarity_matrix[labels.index('3makk2')][labels.index('wisd1')], similarity_matrix[labels.index('3makk2')][labels.index('wisd2')]
]
print(f"Среднее сходство разных авторов (3Makk vs Wisd): {np.mean(diff_pairs):.4f}")
#allow download Y

CLTK message: This part of the CLTK depends upon a spaCy NLP mode.
CLTK message: Allow download of spaCy model ``grc_odycy_joint_sm`` from ``https://huggingface.co/chcaa/grc_odycy_joint_sm/resolve/main/grc_odycy_joint_sm-any-py3-none-any.whl``? [Y/n] 
Y
CLTK message: This part of the CLTK depends upon word embedding models from the NLPL project.
Do you want to download file 'http://vectors.nlpl.eu/repository/20/30.zip' to '/root/cltk_data/grc/embeddings/nlpl/30.zip'? [Y/n] 
Y


100%|██████████| 34.3M/34.3M [00:00<00:00, 54.2MiB/s]



Матрица косинусного сходства (MFW = 100)
             josas1     josas2     jdt1       jdt2       3makk1     3makk2     wisd1      wisd2      testjob1   testjob2  
josas1       1.0000     0.7247     0.5256     0.6297     0.3682     0.4041     0.4801     0.4755     0.5866     0.5721    
josas2       0.7247     1.0000     0.5477     0.5256     0.2359     0.2693     0.3263     0.3392     0.5092     0.4674    
jdt1         0.5256     0.5477     1.0000     0.8376     0.5227     0.6213     0.5091     0.5176     0.5100     0.4818    
jdt2         0.6297     0.5256     0.8376     1.0000     0.5311     0.5646     0.5769     0.5575     0.6516     0.5930    
3makk1       0.3682     0.2359     0.5227     0.5311     1.0000     0.7339     0.4693     0.5522     0.4739     0.4683    
3makk2       0.4041     0.2693     0.6213     0.5646     0.7339     1.0000     0.5147     0.5406     0.4177     0.4831    
wisd1        0.4801     0.3263     0.5091     0.5769     0.4693     0.5147     1.0000     0.7887 

In [3]:
import numpy as np
from scipy.spatial.distance import cosine

v1 = np.array([0.0006042, 0, 0.0009063, 0.0016113, 0.0018127, 0.0007049, 0, 0.0002014, 0.0002014, 0.0004028, 0.0003021, 0.0017120, 0.0001007 ])  # Текст 1
v2 = np.array([0.0017271, 0.0002879, 0.0005757, 0.0002879, 0.0023028, 0.0008636, 0.0002879, 0, 0.0005757, 0, 0, 0.0005757, 0 ])  # Текст 2

cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(f"Косинусное сходство: {cos_sim:.4f}")

Косинусное сходство: 0.7498
